# ADME API smoke test (Databricks — cluster only, no App)

Run this on an **all-purpose / job cluster** in your workspace (e.g. [adb-4173618801742158](https://adb-4173618801742158.18.azuredatabricks.net)). **Databricks Apps are not required.**

**Cluster (managed identity path):** Azure Databricks with **SINGLE_USER** or **NO_ISOLATION**, **Access mode** that supports the workspace **managed identity**, and that identity granted the **ADME API** app role for the correct `CLIENT_ID` / scope.

**Token path (if MI is not available or you prefer a JWT):** set `USE_ENTRA_TOKEN_FROM_SECRET = True`, point `SECRET_SCOPE` / `SECRET_KEY` at a scope you can read (POC workspaces often use a Key Vault–backed scope such as `kv-secrets-adme-sbx` — you need **READ** on the scope and the **secret key name** from your admin), **or** paste a JWT into `MANUAL_BEARER_TOKEN`.

**Run order:** pip cell → config cell → credential + session cell → smoke GETs cell.

In [ ]:
%pip install -q azure-identity requests

In [ ]:
# --- ADME target (pick environment) ---
# "sandbox" = Microsoft ADME sandbox URL. "admedemo_test" = admedemo-test instance (adjust CLIENT_ID if your admin says so).
ADME_ENV = "sandbox"  # "sandbox" | "admedemo_test"

if ADME_ENV == "sandbox":
    baseUrl = "https://admesbxscusins1.energy.azure.com"
elif ADME_ENV == "admedemo_test":
    baseUrl = "https://admedemo-test.energy.azure.com"
else:
    raise ValueError("Set ADME_ENV to 'sandbox' or 'admedemo_test'")

dataPartitionId = "opendes"
TENANT_ID = "72f988bf-86f1-41af-91ab-2d7cd011db47"
# ADME API app (audience) for token scope — confirm for your ADME_ENV in Entra / integration docs
CLIENT_ID = "e37a6c70-7cbc-4593-80fc-01c1f20203f7"

SEISTORE_STATUS_PATH = "/seistore-svc/api/v3/svcstatus"
RESERVOIR_DDMS_HEALTH_PATH = "/api/reservoir-ddms/v2/health/info"
CRS_CATALOG_INFO_PATH = "/api/crs/catalog/v3/info"
ENTITLEMENTS_GROUPS_PATH = "/api/entitlements/v2/groups"
LEGAL_TAGS_PATH = "/api/legal/v1/legaltags?valid=true"
PARTITION_PARTITIONS_PATH = "/api/partition/v1/partitions"
FILE_WELL_KNOWN_PATH = "/api/file/v2/well-known/configuration"
SEARCH_LIVENESS_PATH = "/api/search/v2/liveness"
INDEXER_READINESS_PATH = "/api/indexer/v2/readiness"

# Token from Databricks secret (needs READ on scope). POC example scope name (Key Vault–backed):
USE_ENTRA_TOKEN_FROM_SECRET = False
SECRET_SCOPE = "kv-secrets-adme-sbx"
SECRET_KEY = "REPLACE_WITH_KEY_NAME_FROM_ADMIN"  # vault secret name mapped into the scope
MANUAL_BEARER_TOKEN = ""  # or paste JWT here to skip MI and secrets

SCOPE = f"api://{CLIENT_ID}/.default"
print("ADME_ENV:", ADME_ENV)
print("baseUrl:", baseUrl)
print("dataPartitionId:", dataPartitionId)
print("SCOPE:", SCOPE)

In [ ]:
import json
import time
import base64
from typing import Optional

import requests
from azure.core.credentials import AccessToken, TokenCredential
from azure.identity import ManagedIdentityCredential


class StaticBearerTokenCredential(TokenCredential):
    def __init__(self, token: str, expires_on: Optional[int] = None):
        self._token = token.strip()
        self._expires_on = expires_on or (int(time.time()) + 3600)

    def get_token(self, *scopes, **kwargs):
        return AccessToken(self._token, self._expires_on)


class RefreshableSession(requests.Session):
    def __init__(self, credential: TokenCredential, scope: str, data_partition_id: str):
        super().__init__()
        self.credential = credential
        self.scope = scope
        self.headers.update(
            {"data-partition-id": data_partition_id, "Accept": "application/json"}
        )

    def request(self, method, url, **kwargs):
        tok = self.credential.get_token(self.scope)
        self.headers["Authorization"] = f"Bearer {tok.token}"
        return super().request(method, url, **kwargs)


def _pick_credential():
    if MANUAL_BEARER_TOKEN and MANUAL_BEARER_TOKEN.strip():
        print("Using MANUAL_BEARER_TOKEN")
        return StaticBearerTokenCredential(MANUAL_BEARER_TOKEN)
    if USE_ENTRA_TOKEN_FROM_SECRET:
        tok = dbutils.secrets.get(SECRET_SCOPE, SECRET_KEY)
        print("Using secret", SECRET_SCOPE, "/", SECRET_KEY)
        return StaticBearerTokenCredential(tok)
    print("Using ManagedIdentityCredential (cluster default MI)")
    return ManagedIdentityCredential()


credential = _pick_credential()
t0 = credential.get_token(SCOPE)
print("Token acquired, length:", len(t0.token), "expires_on:", t0.expires_on)

parts = t0.token.split(".")
if len(parts) == 3:
    pad = parts[1] + "=" * (-len(parts[1]) % 4)
    claims = json.loads(base64.urlsafe_b64decode(pad))
    print("aud:", claims.get("aud"), "appid:", claims.get("appid", claims.get("azp")), "roles:", claims.get("roles"))

session = RefreshableSession(credential, SCOPE, dataPartitionId)
print("Session ready.")

### Run smoke GETs

In [ ]:
candidates = [
    (SEISTORE_STATUS_PATH, "Seistore service status"),
    (RESERVOIR_DDMS_HEALTH_PATH, "Reservoir DDMS health/info"),
    (CRS_CATALOG_INFO_PATH, "CRS Catalog info"),
    (ENTITLEMENTS_GROUPS_PATH, "Entitlements groups"),
    (LEGAL_TAGS_PATH, "Legal tags (valid=true)"),
    (PARTITION_PARTITIONS_PATH, "Partition service (list partitions)"),
    (FILE_WELL_KNOWN_PATH, "File service (well-known configuration)"),
    (SEARCH_LIVENESS_PATH, "Search service (liveness)"),
    (INDEXER_READINESS_PATH, "Indexer service (readiness)"),
]

results = []
for path, label in candidates:
    url = baseUrl.rstrip("/") + path
    print("\n" + "=" * 60)
    print("TEST:", label)
    print("GET ", url)
    try:
        resp = session.get(url, timeout=30)
        results.append((label, path, resp.status_code))
        ok = "✅" if resp.status_code == 200 else "❌"
        print(f"STATUS: {ok} {resp.status_code}")
        print("Content-Type:", resp.headers.get("Content-Type", "N/A"))
        if resp.text:
            try:
                body = resp.json()
                out = json.dumps(body, indent=2)
                print(out[:1000] + (f"\n... [{len(out)} chars]" if len(out) > 1000 else ""))
            except Exception:
                print(resp.text[:500])
    except Exception as e:
        results.append((label, path, f"EXCEPTION: {type(e).__name__}"))
        print("Exception:", repr(e))

print("\n" + "=" * 60)
print("\nSummary:")
print(f"{'Test':<35} {'Path':<45} {'Status'}")
print("-" * 90)
for label, path, status in results:
    icon = "✅" if status == 200 else "❌"
    print(f"{label:<35} {path:<45} {icon} {status}")